# 用神经网络识别人体动作

In [1]:
import torch
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
device

device(type='cuda')

In [2]:
# 以下读取数据、切分片段和划分数据集的代码与上面相同
import csv

import numpy as np

torch.set_num_threads(1)

acc = np.load("datasets/moves/signals.npz")["acc"]
with open("datasets/moves/segments.csv") as f:
    segments = [(int(row["person"]), row["activity"], int(row["length"])) for row in csv.DictReader(f)]
names = ["walking", "upstairs", "downstairs", "sitting", "standing", "lying"]

In [3]:
windows, labels, people = [], [], []
# start 是当前这段记录在 acc 中的起始行号
start = 0
for person, activity, length in segments:
    # 片段的起点每次后移 64 个时刻，最后一个片段也要有完整的 128 个时刻
    for i in range(0, length - 127, 64):
        # 取出 128 个时刻的读数，转置成 (3, 128)，每行是一个方向
        windows.append(acc[start + i : start + i + 128].T)
        # 标签是动作在 names 中的序号，再记下片段属于哪位志愿者
        labels.append(names.index(activity))
        people.append(person)
    start += length
x = torch.tensor(np.array(windows), dtype=torch.float32).to(device)
y = torch.tensor(labels).to(device)
people = torch.tensor(people)

test_people = [2, 4, 9, 10, 12, 13, 18, 20, 24]
others = [p for p in range(1, 31) if p not in test_people]
val_people = np.random.default_rng(42).permutation(others)[:3].tolist()
# 按志愿者给每个片段标上 True/False，既不在测试集也不在验证集的片段归训练集
is_test = torch.isin(people, torch.tensor(test_people))
is_val = torch.isin(people, torch.tensor(val_people))
is_train = ~is_test & ~is_val
x_train, y_train = x[is_train], y[is_train]
x_val, y_val = x[is_val], y[is_val]
x_test, y_test = x[is_test], y[is_test]

In [4]:
print(x_train.shape)
print(y_train.shape)
print(x_train.device, y_train.device)

torch.Size([6183, 3, 128])
torch.Size([6183])
cuda:0 cuda:0


In [5]:
import torch.nn as nn
network = torch.nn.Sequential(
    nn.Flatten(), # 形状变成(N, 384)，N是样本数
    nn.Linear(384, 128),
    nn.ReLU(),
    nn.Linear(128, 6),# 共六种类别
)
print(network)
network.to(device)
print(next(network.parameters()).device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(network.parameters(), lr=0.001)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=384, out_features=128, bias=True)
  (2): ReLU()
  (3): Linear(in_features=128, out_features=6, bias=True)
)
cuda:0


In [6]:
for step in range(5001):
    optimizer.zero_grad()
    batch = torch.randint(0, len(x_train), (64,))
    output = network(x_train[batch])
    loss = loss_fn(output, y_train[batch])

    if step in [0, 100, 500, 2000, 5000]:
        print(f"当前轮次{step}的损失是{loss:.3f}")

    loss.backward()
    optimizer.step()

当前轮次0的损失是3.732
当前轮次100的损失是1.053
当前轮次500的损失是0.739
当前轮次2000的损失是0.538
当前轮次5000的损失是0.277


In [8]:
with torch.no_grad():
    predictions = network(x_test).argmax(dim=1)
    print(((predictions == y_test).sum(dim=0) / len(y_test)).item())

0.7621309757232666
